# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

This stage tracks two distinct wallet groups on the full trade stream (BUY + SELL):

- `copyable_group`: wallets that pass base eligibility and strong copyability thresholds.
- `predicting_group`: the remaining eligible wallets (not in copyable) that pass a predictability score threshold.

Notes:
- Group membership is threshold-based (absolute scores), not target-count based.
- Open-buy skill metrics are kept for diagnostics/legacy comparison.
- `selected_wallets_v2.parquet` stores both groups with a `wallet_group` label.
- Stage2 backtest uses only `predicting_group`.


In [6]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [7]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

# METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
# METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
# METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
# METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [8]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-05-31  TRAIN_A_END_DATE=2026-03-21


## Full-stream wallet metrics for copy-trigger selection

Build per-wallet metrics on the full training trade stream (BUY + SELL).
Copyability is considered here (stage1), not in stage0 filtering.


In [12]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
].copy().reset_index(drop=True)

# Load full processed trades for the full-stream path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)

# df_full = df_full[
#     (df_full['avg_price'] <= 1) 
#     # (df_full['side'] == 'BUY') 
#     ()
#     ].copy().reset_index(drop=True)

df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"] = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"] = df_full["quantity"].astype(float)

# Full-stream PnL / notional across BUY + SELL
df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)

df_slice = df_full[df_full["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
del df_slice
print(len(wallet_vol_train))


42222


In [13]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(
    wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'].replace(0, np.nan),
    0,
    1.0,
).fillna(0.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']

# Keep sorting deterministic for downstream filtering/inspection.
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)


In [14]:
wallet_cohorts = {}

In [17]:
print(len(wallet_vol_train))
wallet_vol_train.head()


42222


,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,average_roi,buy_roi,sell_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0xa231231e1e0a4d69fd7e51c3c6e6676b3f692e37,13.75,17,4,17,514.16,124.69,321.45,1.03,1.00,...,106.90,47.02,-0.61,307.14,2.46,1.19,0.00,0.24,1.00,106.90
1,0x3c6385912d528e4a71a5fb47ef269c960ec7a0df,5.17,2,1,20,40.14,376.45,391.36,1.00,1.00,...,99.00,199.00,-1.00,38.05,0.10,23.15,0.06,9.38,1.00,99.00
2,0x4b1d2d479f67a33141d67005acf069e10a08bdfb,NaN,1,1,1,1.52,150.48,150.48,1.00,1.00,...,99.00,99.00,NaN,0.00,0.00,0.00,0.00,99.00,1.00,99.00
3,0x8da0d29ec66508950ad72846ac3164db7cc41c53,NaN,1,1,1,2.50,205.83,205.83,1.00,1.00,...,82.33,82.33,NaN,0.00,0.00,0.00,0.00,82.33,1.00,82.33
4,0xdff348c95aa09169740de5a2d4ac8019bb3f531d,3.74,2,1,2,101.57,94.50,94.50,1.00,1.00,...,61.50,-1.00,124.00,100.00,1.06,100.00,1.06,0.93,1.00,61.50


In [18]:
# Final wallet selection: threshold-based copyable + predicting groups
# COPYABLE_SCORE_MIN = -1
# PREDICTING_SCORE_MIN = -2

candidates = wallet_vol_train.copy()

for col, default in {
    'top10_pnl_pct': np.nan,
    'top_market_abs_pnl_pct': np.nan,
    'market_pnl_hhi': np.nan,
    'positive_bucket_share': np.nan,
}.items():
    if col not in candidates.columns:
        candidates[col] = default

for col in [
    'average_roi', 'median_roi', 'num_buckets', 'num_markets',
    'pnl_volatility', 'max_drawdown_to_pnl',
    'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct', 'market_pnl_hhi',
    'copyable_roi', 'copyable_pnl_factor', 'copyable_pnl', 'total_notional',
    'max_copyable_drawdown_to_copyable_pnl', 'worst5_pnl_pct', 'positive_bucket_share',
]:
    if col in candidates.columns:
        candidates[col] = pd.to_numeric(candidates[col], errors='coerce')

base_mask = (
    (candidates['average_roi'] >= 0.1)
    # & (candidates['median_roi'] >= 0)
    & (candidates['num_buckets'] >= 20)
    & (candidates['num_markets'] >= 15)
    & (candidates['max_drawdown_to_pnl'] <= 0.2)
    & (candidates['top_market_pnl_pct'] < 0.25)
    & (candidates['top5_pnl_pct'] < 0.55)
    & (candidates['top10_pnl_pct'].fillna(0.85) < 0.8)
    & (candidates['top_market_abs_pnl_pct'].fillna(candidates['top_market_pnl_pct']) < 0.25)
    & (candidates['market_pnl_hhi'].fillna(0.20) < 0.20)
    & (candidates['median_dt'].dt.date <= (pd.Timestamp.today().date() - pd.Timedelta(days=30)))
    & (candidates['total_notional'] >= 5_000)
)

base_mask2 = (
    (candidates['buy_roi'] >= 0.1)
    # & (candidates['median_roi'] >= 0)
    & (candidates['num_buckets'] >= 20)
    & (candidates['num_markets'] >= 15)
    & (candidates['max_drawdown_to_pnl'] <= 0.2)
    & (candidates['max_copyable_drawdown_to_copyable_pnl'] <= 0.3)
    # & (candidates['top_market_pnl_pct'] < 0.25)
    # & (candidates['top5_pnl_pct'] < 0.55)
    # & (candidates['top10_pnl_pct'].fillna(0.85) < 0.8)
    & (candidates['market_pnl_hhi'].fillna(0.20) < 0.1)
    & (candidates['median_dt'].dt.date <= (pd.Timestamp.today().date() - pd.Timedelta(days=30)))
    & (candidates['total_notional'] >= 1_000)
)

eligible_base = candidates[base_mask2].copy()
if eligible_base.empty:
    raise ValueError('No wallets passed base eligibility filters.')

eligible_base['sample_mult'] = np.clip(
    np.log1p(eligible_base['num_buckets']) / np.log(2000.0),
    0.20,
    1.00,
)
eligible_base['downside_tail'] = eligible_base['worst5_pnl_pct'].abs().fillna(0.0)
eligible_base['predictability_score'] = eligible_base['sample_mult'] * (
    # 2.5 * eligible_base['average_roi'].fillna(0.0)
    # + 1.5 * eligible_base['median_roi'].fillna(0.0)
    + 1.2 * (eligible_base['positive_bucket_share'].fillna(0.5) - 0.5)
    - 1.0 * eligible_base['pnl_volatility'].fillna(0.0)
    - 1.2 * eligible_base['max_drawdown_to_pnl'].fillna(0.0)
    - 0.8 * eligible_base['top_market_abs_pnl_pct'].fillna(eligible_base['top_market_pnl_pct']).fillna(0.0)
    - 0.6 * eligible_base['market_pnl_hhi'].fillna(0.0)
    - 0.5 * eligible_base['downside_tail']
)
eligible_base['predicting_final_score'] = eligible_base['predictability_score']

copyable_mask = (
    (eligible_base['copyable_pnl'] > 0)
    & (eligible_base['average_roi'] >= 0.04)
    # & (eligible_base['copyable_pnl_factor'] >= 0.1)
    & (eligible_base['copyable_roi'] >= 0.05)
    # & (eligible_base['median_roi'] >= 0.00)
    # & (eligible_base['average_roi'] >= 0.04)
    # & (
    #     eligible_base['max_copyable_drawdown_to_copyable_pnl']
    #     .fillna(eligible_base['max_drawdown_to_pnl'])
    #     .fillna(0.0)
    #     <= 0.3
    # )
)
copyable_candidates = eligible_base[copyable_mask].copy()
copyable_candidates['copyable_efficiency'] = copyable_candidates['copyable_pnl'].fillna(0.0) / (copyable_candidates['total_notional'].fillna(0.0) + 1.0)
copyable_candidates['copyable_dd_ratio'] = (
    copyable_candidates['max_copyable_drawdown_to_copyable_pnl']
    .fillna(copyable_candidates['max_drawdown_to_pnl'])
    .fillna(0.0)
)
copyable_candidates['copyable_score'] = (
    1.8 * copyable_candidates['copyable_roi'].fillna(0.0)
    + 1.2 * copyable_candidates['copyable_pnl_factor'].fillna(0.0)
    + 25.0 * copyable_candidates['copyable_efficiency'].clip(lower=-1.0, upper=1.0)
    - 0.8 * copyable_candidates['copyable_dd_ratio'].clip(lower=0.0)
    - 0.5 * copyable_candidates['top_market_abs_pnl_pct'].fillna(copyable_candidates['top_market_pnl_pct']).fillna(0.0)
)
copyable_candidates['final_score'] = (
    0.60 * copyable_candidates['predictability_score']
    + 0.40 * copyable_candidates['copyable_score']
)

wallet_cohorts['copyable_group'] = (
    copyable_candidates
    .sort_values('final_score', ascending=False)
    .reset_index(drop=True)
)

predicting_pool = eligible_base[
    ~eligible_base['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])
].copy()
wallet_cohorts['predicting_group'] = (
    predicting_pool
    # predicting_pool[predicting_pool['predicting_final_score'] >= PREDICTING_SCORE_MIN]
    .sort_values('predicting_final_score', ascending=False)
    .reset_index(drop=True)
)

if wallet_cohorts['copyable_group'].empty:
    raise ValueError('No wallets passed copyable-group thresholds.')
if wallet_cohorts['predicting_group'].empty:
    raise ValueError('No wallets passed predicting-group threshold.')

overlap = set(wallet_cohorts['copyable_group']['wallet']) & set(wallet_cohorts['predicting_group']['wallet'])
if overlap:
    raise ValueError(f'Wallet groups must be disjoint; found overlap of {len(overlap)} wallets.')

wallet_cohorts['copyable_group']['wallet_quality'] = wallet_cohorts['copyable_group']['final_score']
wallet_cohorts['predicting_group']['wallet_quality'] = wallet_cohorts['predicting_group']['predicting_final_score']

wallet_cohorts['multi_filter'] = wallet_cohorts['copyable_group'].copy()

print(f"Base-eligible wallets: {len(eligible_base):,}")
print(f"Copyable candidates: {len(copyable_candidates):,}")
print(f"Selected wallets (copyable group): {len(wallet_cohorts['copyable_group']):,}")
print(f"Selected wallets (predicting group): {len(wallet_cohorts['predicting_group']):,}")
wallet_cohorts['copyable_group'].head(10)



Base-eligible wallets: 81
Copyable candidates: 49
Selected wallets (copyable group): 49
Selected wallets (predicting group): 32


,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
0,0xa68067f199a367801b50cd131825f7ba19f3ba4a,2.40,28,20,28,2254.74,754.81,781.08,0.99,1.42,...,0.38,0.44,0.46,-1.30,-1.30,0.35,0.12,10.36,3.36,3.36
1,0xbc08db94ad118c943afb66eccce9afb21ee3267a,2.31,41,27,42,2122.59,600.63,533.50,1.01,1.40,...,0.15,0.49,0.47,-1.42,-1.42,0.25,0.12,7.42,2.11,2.11
2,0xf9d969bd75d0ee5e76acb2102a9bbc0efb431ff9,3.41,46,25,47,2830.49,1361.10,762.37,0.94,1.28,...,0.45,0.51,0.26,-1.98,-1.98,0.27,0.09,8.04,2.03,2.03
3,0x6011655c4afb76f36dd1b08a137a1ba73466b31e,0.79,548,188,826,35342.02,4869.92,2161.07,0.22,0.33,...,2.16,0.83,0.01,-0.58,-0.58,0.06,0.02,5.91,2.02,2.02
4,0x4e154de7f8cdb360003803e41274f08d8af80cc5,1.27,72,25,120,2316.31,994.32,393.45,0.70,1.15,...,0.97,0.56,0.25,-0.95,-0.95,0.17,0.30,6.15,1.89,1.89
5,0x76305b2a31e7ec35189650c93e8df2a15b92789d,0.55,351,104,422,4675.42,1380.50,484.21,0.51,0.67,...,1.17,0.77,0.10,-0.67,-0.67,0.10,0.22,4.86,1.54,1.54
6,0x23476c893433c537c56c2479003d8c97dd315adf,1.11,1008,506,1092,4598.47,867.43,857.70,0.94,1.17,...,0.15,0.91,0.09,-1.45,-1.45,0.19,0.06,5.97,1.52,1.52
7,0x490e59912217dc88940c4f879115d67cd65a123a,1.05,431,200,543,3177.94,421.64,395.41,1.14,1.41,...,0.79,0.80,0.43,-1.41,-1.41,0.12,0.18,5.39,1.31,1.31
8,0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,0.83,71,38,80,8009.87,1145.21,687.19,0.49,0.71,...,0.41,0.56,0.24,-0.50,-0.50,0.09,0.11,3.47,1.09,1.09
9,0x653e1e61a51e2ee4ba1e7ff934b4ea994bbfbbc0,4.09,31,26,32,7141.50,2200.93,1291.64,0.85,1.21,...,0.31,0.46,0.39,-2.00,-2.00,0.18,0.11,5.60,1.04,1.04


In [19]:
copyable_candidates.sort_values('trade_count', ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score
9977,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,1.78,6710,2032,7941,635750.66,93396.29,16072.98,0.36,0.46,...,0.17,0.06,1.00,0.10,-1.92,-1.92,0.03,0.28,0.68,-0.88
9561,0x1c144e30f405a25f991cbd8baa15d40599090869,3.66,4349,1048,5728,618405.30,82414.42,27903.99,0.36,0.54,...,0.34,0.07,1.00,0.15,-3.80,-3.80,0.05,0.18,1.49,-1.68
5590,0xe732156a2d84cdfb4de831d3f11a22899e49898f,3.21,3476,698,4783,371104.23,30109.71,19971.55,0.55,0.73,...,0.66,0.27,1.00,0.25,-3.55,-3.55,0.05,0.18,2.42,-1.16
9880,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,0.91,4115,1106,4714,225426.08,29006.48,4236.02,0.20,0.32,...,0.15,0.06,1.00,0.05,-0.89,-0.89,0.02,0.13,0.62,-0.29
9083,0x8fb431f5057112cfdb0e7f566b4b687cb69cb9ed,1.13,1813,359,4251,85467.05,12334.61,4390.94,0.43,0.66,...,0.36,0.08,0.99,0.11,-1.58,-1.58,0.05,0.23,1.64,-0.29
8595,0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,1.66,2662,306,4156,161218.71,12555.48,6080.87,0.28,0.48,...,0.48,0.10,1.00,0.28,-1.98,-1.98,0.04,0.17,1.54,-0.57
7039,0x99984e22205053950eb25453779267bcc1aee858,0.33,3880,1063,4109,33947.65,3698.31,441.62,0.20,0.32,...,0.12,0.17,1.00,0.13,-0.46,-0.46,0.01,0.28,0.54,-0.06
8908,0x5f4fc359460dbe427ae7d4f81b4a34b3678cc2f1,1.39,2227,580,2755,34036.46,6870.42,4090.39,0.58,0.83,...,0.60,0.09,1.00,0.11,-1.80,-1.80,0.12,0.12,3.68,0.39
4759,0x736539924a5602b37a03a54fc12c1cc8f98964da,1.87,1946,248,2714,278823.02,17923.29,7409.89,0.30,0.48,...,0.41,0.36,1.00,0.26,-2.17,-2.17,0.03,0.22,1.58,-0.67
8356,0x7231a52f9de4fda5218d0e63f30a3499a4535afe,1.25,1158,149,2327,30066.42,5386.54,1138.09,0.51,0.85,...,0.21,0.11,0.93,0.28,-1.73,-1.73,0.04,0.23,1.14,-0.58


In [20]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [21]:
filter_wallets = set(wallet_cohorts['multi_filter']["wallet"])
df_test =  df_full[
    (df_full['dt'] > pd.to_datetime("2026-04-16", utc=True))
    # & (df_full['wallet'] == '0xbacd00c9080a82ded56f504ee8810af732b0ab35')
    & (df_full['wallet'].isin(filter_wallets))
    ].groupby(['question', 'condition_id', 'outcome']).agg(
    # ].groupby(['wallet']).agg(
        pnl_sum=('pnl', 'sum'),
        trade_count=('pnl', 'size'),
        wallet_count=('wallet', 'nunique'),
    ).reset_index().sort_values('pnl_sum', key=abs, ascending=False)

big_conditions = df_test.groupby('condition_id').agg(
    abs_pnl_sum=('pnl_sum', lambda x: abs(x).sum()),
    min_abs_pnl_pct=('pnl_sum', lambda x: abs(x).min() / abs(x).sum() if abs(x).sum() > 0 else 0)
).reset_index().sort_values('abs_pnl_sum', ascending=False)



print(df_test['pnl_sum'].sum())
df_test = df_test[df_test['condition_id'].isin(big_conditions['condition_id'])].sort_values('question', ascending=False).merge(big_conditions, on='condition_id', how='left')

print(df_test[df_test['min_abs_pnl_pct']<0.1]['pnl_sum'].sum()) 

df_test[df_test['min_abs_pnl_pct'] < 0.1].sort_values(['abs_pnl_sum'], ascending=False).head(20)

246865.20164999997
88194.184223


,question,condition_id,outcome,pnl_sum,trade_count,wallet_count,abs_pnl_sum,min_abs_pnl_pct
6950,"Will Trump say ""Iran"" during events with Xi Ji...",0xaa145cf5714455f546afe53037b8712df749ba96c048...,Yes,0.00,2,1,6816.97,0.00
6949,"Will Trump say ""Iran"" during events with Xi Ji...",0xaa145cf5714455f546afe53037b8712df749ba96c048...,No,6816.97,31,5,6816.97,0.00
8177,US announces new Iran agreement/ceasefire exte...,0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198...,No,-39.39,15,3,6078.24,0.01
8176,US announces new Iran agreement/ceasefire exte...,0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198...,Yes,6038.85,98,5,6078.24,0.01
8108,"US x Iran diplomatic meeting by June 19, 2026?",0xa6e388d87a81a17dd8f2046479e11c50bf32bcf475ce...,No,4889.23,48,5,5080.51,0.04
8107,"US x Iran diplomatic meeting by June 19, 2026?",0xa6e388d87a81a17dd8f2046479e11c50bf32bcf475ce...,Yes,-191.28,11,2,5080.51,0.04
7252,Will Trump agree to Iranian enrichment of uran...,0x807ba575f84169bdb061dd05da3e8eb314a129aa152e...,No,-0.06,2,1,5024.48,0.00
7251,Will Trump agree to Iranian enrichment of uran...,0x807ba575f84169bdb061dd05da3e8eb314a129aa152e...,Yes,-5024.42,8,1,5024.48,0.00
8421,Iran agrees to surrender enriched uranium stoc...,0xd39905267d79b715f078279ef41f311b791c6e2c7361...,No,4546.13,8,4,4724.75,0.04
8422,Iran agrees to surrender enriched uranium stoc...,0xd39905267d79b715f078279ef41f311b791c6e2c7361...,Yes,178.62,11,2,4724.75,0.04


In [22]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False).head(10)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,buy_roi,sell_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
48,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,10.59,1052,290,1874,1183401.66,163317.51,46422.23,0.44,0.66,-0.26,0.09,0.07,0.03,0.61,0.05,2025-12-19 22:17:30+00:00,0.76,0.14,0.15,13660.97,0.08,10133.74,0.22,0.14,0.28,0.22,0.92,0.26,-9.86,-9.86,0.04,0.22,1.50,-5.32,-5.32
46,0x1c144e30f405a25f991cbd8baa15d40599090869,3.66,4349,1048,5728,618405.30,82414.42,27903.99,0.36,0.54,-0.15,0.10,0.04,0.01,0.60,0.12,2026-01-27 20:30:00+00:00,0.20,0.16,0.05,10118.78,0.12,4941.62,0.18,0.13,0.34,0.07,1.00,0.15,-3.80,-3.80,0.05,0.18,1.49,-1.68,-1.68
41,0x88b59d79b6e1659c95a0043028e5bb7a26e6205c,4.19,300,36,647,475683.39,59932.81,23172.65,0.44,0.67,-0.15,0.18,0.17,0.08,0.84,0.16,2025-06-22 17:45:00+00:00,0.19,0.15,-0.97,2775.43,0.05,1690.00,0.07,0.13,0.39,0.07,0.75,0.15,-3.08,-3.08,0.05,0.07,1.67,-1.18,-1.18
40,0xe732156a2d84cdfb4de831d3f11a22899e49898f,3.21,3476,698,4783,371104.23,30109.71,19971.55,0.55,0.73,-0.25,0.25,0.13,0.03,0.56,0.01,2026-01-14 14:07:30+00:00,0.41,0.18,-0.07,3972.61,0.13,3524.96,0.18,0.08,0.66,0.27,1.00,0.25,-3.55,-3.55,0.05,0.18,2.42,-1.16,-1.16
37,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,1.78,6710,2032,7941,635750.66,93396.29,16072.98,0.36,0.46,-0.10,0.20,0.06,0.01,0.56,0.06,2026-01-24 23:12:30+00:00,0.33,0.17,0.08,8553.27,0.09,4556.81,0.28,0.15,0.17,0.06,1.00,0.10,-1.92,-1.92,0.03,0.28,0.68,-0.88,-0.88
13,0x78a19803306d350cd1120bb07fcfe91a665be6bd,8.02,187,93,392,29172.11,13022.22,11349.30,0.98,1.30,-0.39,0.19,0.10,0.03,0.42,-1.00,2026-04-15 14:35:00+00:00,0.21,1.25,-0.39,1751.94,0.13,1535.55,0.14,0.45,0.87,0.18,0.69,0.39,-5.91,-5.91,0.39,0.14,10.94,0.83,0.83
47,0xd1a535ed8543fc1852c87e2fb9ef6fed1c549de7,7.17,909,58,1054,70577.81,43172.62,10920.75,0.64,0.84,-0.13,0.29,0.18,0.10,0.26,-1.00,2025-11-28 11:35:00+00:00,0.86,1.06,0.09,5593.02,0.13,629.07,0.06,0.61,0.25,0.22,0.90,0.13,-7.06,-7.06,0.15,0.06,4.43,-2.47,-2.47
42,0xf989bd9c62b1eae2c388515fcc766527a8b147cc,5.65,84,36,97,118069.01,58540.33,7487.91,0.36,0.57,-0.04,0.17,0.16,0.08,0.84,0.46,2025-03-07 20:47:30+00:00,0.63,0.51,-0.96,1386.68,0.02,93.57,0.01,0.50,0.13,0.08,0.58,0.04,-3.19,-3.19,0.06,0.01,1.79,-1.20,-1.20
35,0x736539924a5602b37a03a54fc12c1cc8f98964da,1.87,1946,248,2714,278823.02,17923.29,7409.89,0.30,0.48,-0.26,0.14,0.08,0.02,0.58,0.03,2026-04-03 12:07:30+00:00,0.87,0.13,-0.02,2909.60,0.16,1660.14,0.22,0.06,0.41,0.36,1.00,0.26,-2.17,-2.17,0.03,0.22,1.58,-0.67,-0.67
36,0x27b820e5203aa114acc2712e0e1d0ad758abb68c,2.23,910,252,1471,231184.05,27765.60,7349.28,0.44,0.68,-0.06,0.13,0.11,0.05,0.59,0.00,2025-10-13 16:25:00+00:00,0.24,0.16,-0.04,1052.59,0.04,1074.73,0.15,0.12,0.26,0.06,0.90,0.06,-2.08,-2.08,0.03,0.15,1.06,-0.82,-0.82


In [23]:
wallets = set(wallet_cohorts['copyable_group']['wallet'])

df = df_full #[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

33706813
23703


In [31]:
df_wallets[df_wallets['side'] == 'BUY'].groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    copyable_pnl=('copyable_pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique())
).sort_values(by="copyable_pnl", key=np.abs, ascending=False).merge(mdf[['condition_id', 'question']], on="condition_id", how="left").head(20)

,condition_id,pnl,copyable_pnl,notional,sell_count,buy_count,wallets,question
0,0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198...,12285.83,6305.58,4217.31,0,57,6,US announces new Iran agreement/ceasefire exte...
1,0xab1209b73e1ce9ebb4905dad781496385b5102c04888...,6386.72,5240.90,5342.66,0,84,6,US announces new Iran agreement/ceasefire exte...
2,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e...,2910.81,5003.60,19106.74,0,111,7,"US x Iran permanent peace deal by June 15, 2026?"
3,0xd39905267d79b715f078279ef41f311b791c6e2c7361...,4549.11,4554.70,1493.82,0,10,5,Iran agrees to surrender enriched uranium stoc...
4,0x3094a2b925483a06aa72945a1472e311e5eb6be75284...,2616.54,2633.65,361.45,0,5,3,Will Donald Trump announce that the United Sta...
5,0x46dd571d8c32bd58dcc553ae6c2a1ae8dda6d1ec6994...,4327.67,1700.33,5287.56,0,27,3,Will the text of the US-Iran agreement be rele...
6,0x271660fedc0b7e2fb1d7f007b704e67102cb29d3def2...,5012.41,1630.98,5446.64,0,90,1,Will SpaceX's valuation hit (HIGH) $3.0T by Ju...
7,0xd8664606498aba2a8ccab56656860cb3f02d74de80e9...,-1278.51,-1423.13,5087.04,0,41,6,Israel closes its airspace by June 15?
8,0xbeab83146920be8f52b52fdbcb3464b24dc553f7b47e...,-1928.06,-1268.35,3377.69,0,7,3,US-Iran nuclear deal by May 31?
9,0x25fe4f0d623da8b9202314313bff75da3bb519fd36d4...,-1470.00,-1176.97,1470.00,0,1,1,Will the highest temperature in Toronto be 22°...


In [32]:
wallet_set = set(wallet_cohorts['copyable_group']['wallet'])
print(f"Copyable wallets: {len(wallet_set)} with fills {df_wallets[df_wallets['wallet'].isin(wallet_set)].shape[0]} with total PnL: {df_wallets[df_wallets['wallet'].isin(wallet_set)]['pnl'].sum():,.2f}")
df_wallets[df_wallets['wallet'].isin(wallet_set)]['trade_pnl'].sum()

Copyable wallets: 49 with fills 23703 with total PnL: 125,170.17


np.float64(125170.172514)

In [33]:
MARKET = '0xaabf22a03217285585592f3082f27e2b0f4c1cee2bb4003e84817100bbcf1938'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty=('copyable_qty', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: condition_id       0xaabf22a03217285585592f3082f27e2b0f4c1cee2bb4...
end_date_iso                                    2026-07-02T00:00:00Z
question                         S&P 500 (SPX) Up or Down on July 2?
tags               [Finance, Up or Down, Hide From New, Daily, In...
primary_tag                                                  Finance
winner_token_id    2351022491940408504984084438472877165257561395...
outcome                                                           Up
Name: 189681, dtype: object


In [34]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [35]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

49


,wallet,primary_tag,primary_tag_qty,primary_tag_ratio,pnl
125,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,Politics,2065020.16,0.89,143508.71
218,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,Politics,1418521.44,0.91,91827.70
40,0x1c144e30f405a25f991cbd8baa15d40599090869,Politics,1208168.09,0.89,80717.94
481,0xf989bd9c62b1eae2c388515fcc766527a8b147cc,Politics,162341.81,0.86,49868.54
306,0x88b59d79b6e1659c95a0043028e5bb7a26e6205c,Politics,456158.98,0.69,47743.81
204,0x6011655c4afb76f36dd1b08a137a1ba73466b31e,Weather,1010244.94,1.00,43612.38
99,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,Politics,292981.48,0.75,20971.10
24,0x13faff6b4dbabaaf9fd780befcd13dcb027a3fac,Esports,117145.93,0.99,20780.07
453,0xe732156a2d84cdfb4de831d3f11a22899e49898f,Politics,484380.29,0.56,17317.33
298,0x86a642f6ee9888795c62a43ec9ca175927bc1ac6,Politics,132754.58,0.96,16984.35


In [38]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [39]:
import plotly.graph_objects as go

print(f"copyable_group wallets: {len(wallet_cohorts['copyable_group']):,}")
print(f"predicting_group wallets: {len(wallet_cohorts['predicting_group']):,}")

pd.set_option('display.max_rows', 100)

copyable_view = wallet_cohorts['copyable_group'].sort_values('final_score', ascending=False).head(20)
predicting_view = wallet_cohorts['predicting_group'].sort_values('predicting_final_score', ascending=False).head(20)

copyable_view, predicting_view

plot_start = pd.Timestamp(END_DATE_TRAIN, tz='UTC') + pd.Timedelta(days=1)
plot_rows = []
for group_label, cohort_name in [('copyable', 'copyable_group'), ('predicting', 'predicting_group')]:
    wallets = set(wallet_cohorts[cohort_name]['wallet'])
    group_trades = df_full[
        (df_full['wallet'].isin(wallets))
        & (df_full['dt'] >= plot_start)
    ][['dt', 'pnl', 'copyable_pnl']].copy()

    daily = (
        group_trades
        .assign(plot_dt=group_trades['dt'].dt.floor('D'))
        .groupby('plot_dt', as_index=False)[['pnl', 'copyable_pnl']]
        .sum()
        .sort_values('plot_dt')
        .reset_index(drop=True)
    )
    daily['cum_trade_pnl'] = daily['pnl'].cumsum()
    daily['cum_copyable_pnl'] = daily['copyable_pnl'].cumsum()
    daily['wallet_group'] = group_label
    plot_rows.append(daily)

group_perf = pd.concat(plot_rows, ignore_index=True)

fig = go.Figure()
colors = {'copyable': '#1f77b4', 'predicting': '#ff7f0e'}
for group_label in ['copyable', 'predicting']:
    g = group_perf[group_perf['wallet_group'] == group_label]
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_trade_pnl'],
            mode='lines',
            name=f'{group_label} trade pnl',
            line=dict(color=colors[group_label], width=2),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_copyable_pnl'],
            mode='lines',
            name=f'{group_label} copyable pnl',
            line=dict(color=colors[group_label], width=2, dash='dash'),
        )
    )

fig.update_layout(
    title='Stage1 wallet-group performance on test period',
    xaxis_title='Date',
    yaxis_title='Cumulative PnL (USDC)',
    template='plotly_white',
)
fig.show(renderer='browser')


copyable_group wallets: 49
predicting_group wallets: 32


In [40]:
copyable_wallets = wallet_cohorts['copyable_group'].copy().assign(wallet_group='copyable')
predicting_wallets = wallet_cohorts['predicting_group'].copy().assign(wallet_group='predicting')

selected_wallets = (
    pd.concat([copyable_wallets, predicting_wallets], ignore_index=True)
    .drop_duplicates(subset=['wallet', 'wallet_group'])
    .reset_index(drop=True)
)

selected_wallets.to_parquet(WORKSPACE_DIR / 'selected_wallets_v2.parquet', index=False)
print(f"Saved selected wallets: {len(selected_wallets):,}")
print(selected_wallets['wallet_group'].value_counts().to_dict())


Saved selected wallets: 81
{'copyable': 49, 'predicting': 32}


In [41]:
WORKSPACE_DIR / "selected_wallets_v2.parquet"


PosixPath('../../data/trade_signals_workspace_v2/selected_wallets_v2.parquet')

## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [42]:
from polymarket_analysis.strategy.definition import TriggerSpec, WalletSelectionStrategy
from polymarket_analysis.strategy.registry import save_strategy

SIGNAL_THRESHOLD = 0.65

selected_wallets = wallet_cohorts['predicting_group'].copy().reset_index(drop=True)
predicting_wallets = selected_wallets.copy()
if 'wallet_quality' not in predicting_wallets.columns:
    if 'predicting_final_score' in predicting_wallets.columns:
        predicting_wallets['wallet_quality'] = predicting_wallets['predicting_final_score']
    else:
        predicting_wallets['wallet_quality'] = 0.0

strategy_specs = [
    (
        'predicting_group__score_threshold',
        f'predicting_group | score >= {SIGNAL_THRESHOLD:.2f} (Kelly)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.score_threshold',
            params={'threshold': SIGNAL_THRESHOLD, 'dynamic_sizing': True},
            mode='frame',
        ),
    ),
    (
        'predicting_group__all_open_buys',
        'predicting_group | all open-buys (fixed stake)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.all_open_buys',
            params={'dynamic_sizing': False},
            mode='frame',
        ),
    ),
    (
        'predicting_group__copy_triggers',
        'predicting_group | copy all triggers (tight slippage)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.copy_triggers',
            params={'dynamic_sizing': False},
            mode='frame',
        ),
    ),
]

all_strategies = []
for strategy_id, name, trigger in strategy_specs:
    all_strategies.append(
        WalletSelectionStrategy(
            strategy_id=strategy_id,
            name=name,
            selection_method='predicting_group',
            trigger=trigger,
            wallets=predicting_wallets.copy(),
            params={
                'selection_metric': 'predicting_final_score',
                'top_n': len(predicting_wallets),
                'signal_threshold': SIGNAL_THRESHOLD if trigger.fn_ref.endswith('score_threshold') else None,
            },
            metadata={
                'cohort': 'predicting_group',
                'wallet_group': 'predicting',
                'selection_metric': 'predicting_final_score',
                'signal_threshold': SIGNAL_THRESHOLD,
                'top_n': len(predicting_wallets),
                'end_date_train': str(END_DATE_TRAIN),
                'train_a_end_date': str(TRAIN_A_END_DATE),
                'selection_notes': (
                    'threshold-based: copyable group uses copyability quality gates, '
                    'predicting group uses predictability threshold on remaining wallets'
                ),
            },
        )
    )

for strategy in all_strategies:
    parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
    print(f"Saved [{strategy.strategy_id}] wallets={len(strategy.wallets):3d} trigger={strategy.trigger.fn_ref.split('.')[-1]}")

print(f'\nTotal strategies saved: {len(all_strategies)}')


Saved [predicting_group__score_threshold] wallets= 32 trigger=score_threshold
Saved [predicting_group__all_open_buys] wallets= 32 trigger=all_open_buys
Saved [predicting_group__copy_triggers] wallets= 32 trigger=copy_triggers

Total strategies saved: 3


## Strategy registry summary

In [43]:
from polymarket_analysis.strategy.registry import load_all_strategies

registry = load_all_strategies(WORKSPACE_DIR)
summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        'strategy_id': s.strategy_id,
        'name': s.name,
        'selection_method': s.selection_method,
        'num_wallets': len(s.wallets),
        'trigger_fn': s.trigger.fn_ref.split('.')[-1],
        'threshold': s.trigger.params.get('threshold'),
        'dynamic_sizing': s.trigger.params.get('dynamic_sizing'),
    })

pd.DataFrame(summary_rows).sort_values('strategy_id').reset_index(drop=True)


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__al...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | a...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,all_open_buys,NaN,False
1,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__co...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | c...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,copy_triggers,NaN,False
2,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__sc...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | s...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,score_threshold,0.65,True
3,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__al...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | a...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,all_open_buys,NaN,False
4,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__co...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | c...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,copy_triggers,NaN,False
5,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__sc...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | s...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,score_threshold,0.65,True
6,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__al...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | a...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,all_open_buys,NaN,False
7,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__co...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | c...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,copy_triggers,NaN,False
8,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__sc...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | s...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,score_threshold,0.65,True
9,0x64e86fefd50cdb6259eb00ff146d9aac03497819__al...,0x64e86fefd50cdb6259eb00ff146d9aac03497819 | a...,0x64e86fefd50cdb6259eb00ff146d9aac03497819,1,all_open_buys,NaN,False


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [44]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [45]:
df_fills = df_full.copy()
df_fills['copyable_pnl_exposure'] = np.where( 
    df_fills['copyable_qty'] > 0,
    df_fills['price'] * df_fills['copyable_qty'],
    np.nan
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

len(df_fills), len(df_fills[df_fills["dt"] < split_dt]), len(df_fills[df_fills["dt"] >= split_dt])

(33706813, 23865975, 9840838)

In [46]:
markets = df_fills[(df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))][['condition_id', 'tags', 'primary_tag']]
markets = markets.groupby('condition_id').agg(
    tags=('tags', 'first'),
    primary_tag=('primary_tag', 'first'),
).reset_index()
markets['has_mentions'] = markets['tags'].apply(lambda tags: 'Mentions' in tags)
mention_markets = set(markets[markets['has_mentions']]['condition_id'])
del markets
len(mention_markets)

4120

In [47]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    & (df_fills['primary_tag'] == 'Politics')
    & (~df_fills['condition_id'].isin(mention_markets))
    & (df_fills["dt"] >= split_dt)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
filter_fills = filter_fills[filter_fills['copyable_qty'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

2503
1383


,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,copyable_pnl_exposure
0,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,0x0e037428a0fe3a7e1652372cab2b4e3e26f83ed0e9f7...,1057908117937573184424868236895185320726203130...,2026-06-11 18:03:24+00:00,BUY,234.69,234.69,0.99,232.33,234.69,2.36,0.07,True,1.00,2026-06-11 18:52:01+00:00,0xbc45987f61f5c77a9e2165e842648678b40f37d3898d...,2,False,6.77,0.07,2.00,2026-12-31T00:00:00Z,Will Trump announce Chris Stewart as the next ...,"[Politics, Trump, Trump Cabinet, Tulsi Gabbard...",Politics,1057908117937573184424868236895185320726203130...,No,2.36,232.33,6.70
1,0x490e59912217dc88940c4f879115d67cd65a123a,0x0200743b501bb76457156e98736b1fd5f65835af8290...,1324549647609819264744950241379355589496814917...,2026-07-05 16:17:44+00:00,BUY,68.15,31.09,0.33,10.26,31.09,20.83,20.83,True,1.00,2026-07-09 22:58:04+00:00,0x453e20a42d72a095b87266bf0b5ba9d10c291ff27b5c...,1,False,31.09,51.76,10.00,2026-07-04T00:00:00Z,Will the White House call a full lid by 6:30PM...,"[Politics, Trump, Lid, Trump Daily]",Politics,1324549647609819264744950241379355589496814917...,No,20.83,10.26,10.26


In [48]:
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 1000)

In [49]:
import numpy as np

(
    filter_fills[
        (filter_fills['condition_id'] == '0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423')
        # (filter_fills["side"] == "BUY")
        & (filter_fills["copyable_qty"] > 0)
        ]
    .assign(price_bucket=lambda df: np.floor(df["price"] * 10) / 10)
    .groupby(['condition_id', 'wallet', 'price_bucket'], as_index=False)
    .agg(
        copyable_pnl=("copyable_pnl", "sum"),
        trades=("condition_id", "count"),
    ).sort_values(by="copyable_pnl", key=abs, ascending=False).reset_index()
)

,index,condition_id,wallet,price_bucket,copyable_pnl,trades
0,11,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0xe732156a2d84cdfb4de831d3f11a22899e49898f,0.20,1843.05,5
1,3,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0x736539924a5602b37a03a54fc12c1cc8f98964da,0.70,1089.53,12
2,4,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0x736539924a5602b37a03a54fc12c1cc8f98964da,0.80,790.44,6
3,2,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0x736539924a5602b37a03a54fc12c1cc8f98964da,0.60,729.22,1
4,8,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,0.20,701.89,9
5,17,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0xf2bbd4a5a02780c14b7ec16a52a3e21a744cae21,0.80,-538.34,6
6,7,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,0.10,530.85,14
7,0,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0x6d16c33b540069f5aa0d9f705b0d759dec927228,0.80,-446.50,1
8,1,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0x736539924a5602b37a03a54fc12c1cc8f98964da,0.50,409.00,1
9,13,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0xe732156a2d84cdfb4de831d3f11a22899e49898f,0.70,304.40,7


In [50]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor('5min')

MAX_EXPOSURE = 100

df = filter_fills.groupby(['bucket', 'condition_id', 'wallet', 'side']).agg(
    copyable_pnl_exposure=('copyable_pnl_exposure', 'sum'),
    total_copyable_qty=('copyable_qty', 'sum'),
    trade_pnl =('trade_pnl', 'sum'),
    copyable_pnl = ('copyable_pnl', 'sum'),
    trades=('condition_id', 'count'),
    copyable_qty=('copyable_qty', 'sum'),
    wallets=('wallet', lambda x: x.nunique()),
).reset_index()

scale = np.minimum(1, MAX_EXPOSURE / df["copyable_pnl_exposure"].replace(0, np.nan))

df = df.assign(
    scaled_copyable_pnl_exposure = df["copyable_pnl_exposure"] * scale,
    scaled_copyable_pnl = df["copyable_pnl"] * scale,
    scaled_copyable_qty = df["copyable_qty"] * scale,
)

df.head(10)

,bucket,condition_id,wallet,side,copyable_pnl_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,copyable_qty,wallets,scaled_copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_qty
0,2026-06-01 09:05:00+00:00,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,BUY,1.39,10.71,9.32,9.32,1,10.71,1,1.39,9.32,10.71
1,2026-06-01 09:10:00+00:00,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,BUY,0.45,3.45,3.00,3.00,1,3.45,1,0.45,3.00,3.45
2,2026-06-01 09:30:00+00:00,0xb920ee2065d4cb86870ef2ab920cfd94f166e3c1f44f614105e37b4b087fb7f3,0x1c144e30f405a25f991cbd8baa15d40599090869,BUY,4.81,5.05,14.25,0.24,1,5.05,1,4.81,0.24,5.05
3,2026-06-01 13:25:00+00:00,0x9a01f6522bb53657564b8934bd946e7cfcef0c4188f2f7311d8a00f3d042ab9f,0x5f4fc359460dbe427ae7d4f81b4a34b3678cc2f1,BUY,0.17,1.00,-0.17,-0.17,1,1.00,1,0.17,-0.17,1.00
4,2026-06-01 13:30:00+00:00,0x9a01f6522bb53657564b8934bd946e7cfcef0c4188f2f7311d8a00f3d042ab9f,0x5f4fc359460dbe427ae7d4f81b4a34b3678cc2f1,BUY,11.39,67.00,-11.39,-11.39,2,67.00,1,11.39,-11.39,67.00
5,2026-06-01 14:10:00+00:00,0xfa0086d4c1ca2f61c2b7ffdb2cba5ae91588ca601afe27909a000ca7f456cbf7,0x86a642f6ee9888795c62a43ec9ca175927bc1ac6,BUY,6.58,10.01,71.66,3.43,1,10.01,1,6.58,3.43,10.01
6,2026-06-01 15:45:00+00:00,0xe6af8954a7ac6db0fda6a2fdb1449b97f40596fff0e1a3e3f81890ac227de92f,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,BUY,7.33,18.33,11.00,11.00,11,18.33,1,7.33,11.00,18.33
7,2026-06-01 18:20:00+00:00,0xbdb44ce1624dbf80fe70a96e7cb027b854aa44c52cf87a6bd0bc9ea2c332c3dc,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,BUY,383.18,788.70,514.16,405.52,1,788.70,1,100.00,105.83,205.83
8,2026-06-01 19:15:00+00:00,0xf526b2fbff94ae996818e76c8b54cc99ffbfa0a1a9972a48253ada65e32786f7,0x736539924a5602b37a03a54fc12c1cc8f98964da,BUY,0.13,2.68,-10.81,-0.13,1,2.68,1,0.13,-0.13,2.68
9,2026-06-01 20:35:00+00:00,0xf526b2fbff94ae996818e76c8b54cc99ffbfa0a1a9972a48253ada65e32786f7,0x736539924a5602b37a03a54fc12c1cc8f98964da,BUY,2.60,50.00,-6.11,-2.60,1,50.00,1,2.60,-2.60,50.00


In [51]:
df = (df.merge(mdf, on='condition_id', how='inner')
    .sort_values(['bucket', 'copyable_pnl_exposure'], ascending=[True, False])
    .reset_index(drop=True)
)
df['end_date_iso'] = pd.to_datetime(df['end_date_iso'], utc=True)
df = df[df['end_date_iso'] > split_dt].copy().reset_index(drop=True)

In [52]:
df_plot = df.groupby('end_date_iso').agg(
    scaled_copyable_pnl=('scaled_copyable_pnl', 'sum'),
    ending_exposure=('scaled_copyable_pnl_exposure', 'sum'),
).reset_index()

df.sort_values('bucket', inplace=True)
df['scaled_copyable_pnl_cumsum'] = df['scaled_copyable_pnl'].cumsum()

df_pnl = df[['bucket', 'scaled_copyable_pnl_cumsum']].copy()

# exposure ends at EOD of end_date, so shift by 24h
df_exposure = df_plot[['end_date_iso', 'ending_exposure']].copy()
df_exposure['end_date_iso'] = pd.to_datetime(df_exposure['end_date_iso']) + pd.Timedelta(days=1)
df_exposure.rename(columns={'end_date_iso': 'dt', 'ending_exposure': 'exposure_delta'}, inplace=True)

resolution_exposure = df_exposure.groupby('dt').agg(exposure_delta=('exposure_delta', 'sum')).reset_index()
resolution_exposure['exposure_delta'] = resolution_exposure['exposure_delta'] * -1

buy_exposure = df[['bucket', 'scaled_copyable_pnl_exposure']].rename(columns={'bucket': 'dt', 'scaled_copyable_pnl_exposure': 'exposure_delta'})

df_exposure = ( pd.concat([resolution_exposure, buy_exposure], ignore_index=True)
    .reset_index(drop=True)
)

df_exposure.sort_values('dt', inplace=True)
df_exposure['exposure'] = df_exposure['exposure_delta'].cumsum()

# df_plot['end_date_iso'] = pd.to_datetime(df_plot['end_date_iso']) + pd.Timedelta(days=1)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_exposure['dt'],
    y=df_exposure['exposure'],
    mode='lines',
    name='exposure'
))

fig.add_trace(go.Scatter(
    x=df_pnl['bucket'],
    y=df_pnl['scaled_copyable_pnl_cumsum'],
    mode='lines',
    name='scaled_copyable_pnl_cumsum'
))

fig.show()

In [53]:
import plotly.express as px

# Aggregate by bucket
bucket_pnl = df.groupby('bucket')['scaled_copyable_pnl'].sum().reset_index()
bucket_pnl['scaled_copyable_pnl'] = bucket_pnl['scaled_copyable_pnl'].cumsum()

# Plot with Plotly
fig = px.line(
    bucket_pnl,
    x='bucket',
    y='scaled_copyable_pnl',
    title='Total Scaled Copyable PnL by Time Bucket',
    labels={'bucket': 'Time Bucket', 'scaled_copyable_pnl': 'Scaled Copyable PnL'}
)
fig.show()

In [54]:
( 
    df.groupby('condition_id').agg(
        total_copyable_exposure=('copyable_pnl_exposure', 'sum'),
        total_copyable_qty=('copyable_qty', 'sum'),
        trade_pnl =('trade_pnl', 'sum'),
        copyable_pnl = ('copyable_pnl', 'sum'),
        trades=('condition_id', 'count'),
        wallets=('wallet', lambda x: x.nunique()),
 ).sort_values('copyable_pnl', ascending=False).head(10)
)

,total_copyable_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,wallets
condition_id,,,,,,
0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,13033.22,21548.71,5853.85,5536.89,48,6
0xab1209b73e1ce9ebb4905dad781496385b5102c048889502b174ed733de1506a,3661.37,11385.00,5581.09,5240.88,25,5
0x3094a2b925483a06aa72945a1472e311e5eb6be75284f61e0c008e279508ddf6,274.41,10154.02,2693.59,2693.59,2,2
0x46dd571d8c32bd58dcc553ae6c2a1ae8dda6d1ec69943bc0f771ea5c35132ca5,2853.89,5768.59,2774.93,1700.33,13,3
0xa6e388d87a81a17dd8f2046479e11c50bf32bcf475ce75a3506c5cf19a09613c,940.93,2028.83,3513.79,787.54,15,5
0x5cfb6cf6857089dc6fa78cfa72c1b3e3f8331d42449407bf4d5dba1232132548,180.86,924.36,1195.51,743.50,2,1
0xbdb44ce1624dbf80fe70a96e7cb027b854aa44c52cf87a6bd0bc9ea2c332c3dc,611.68,1301.33,1572.38,689.65,3,1
0x7b85f02daf8094bb355685cb5697734396dc42c81a0ad16ff9f68131780d4eed,138.03,819.36,744.30,681.33,1,1
0x20af55ab35186377b81219db6cb8615240cba42cea41731091be9484a5f5b122,585.25,1459.27,1533.56,649.03,8,2


In [55]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

else:
    print("PLOT_WALLET_PNL=False — skipping plots")